# J7Scope community trace capture (Colab GPU)

Capture one custom Trace Schema v1 artifact with a real open-weight model, then download a validated ZIP for a gallery pull request.

> **Preview boundary:** the default Qwen2.5-1.5B model and small stochastic Jacobian budget prove the capture pipeline; they do not establish a research result. The output is marked `preview: true`. Runtime and GPU type depend on Colab availability.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/j7scope')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/arthurpanhku/j7scope.git', str(REPO)
    ], check=True)
os.chdir(REPO)
print(f'Working in {REPO}')

In [ ]:
%pip install -q -e .

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Choose Runtime > Change runtime type > GPU, then reconnect.')
props = torch.cuda.get_device_properties(0)
print({
    'gpu': props.name,
    'vram_gib': round(props.total_memory / 1024**3, 2),
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'selected_dtype': 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16',
})

## Configure one trace

Use a unique lowercase `trace_id`. Do not put personal, confidential, copyrighted, or credential-bearing text in the prompt.

In [ ]:
TRACE_ID = 'community-deception-en' # @param {type:'string'}
LANGUAGE = 'en' # @param ['en', 'zh', 'other']
CONCEPT = 'deception' # @param {type:'string'}
PROMPT = 'In one sentence, explain why deception can be tempting.' # @param {type:'string'}
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct' # @param {type:'string'}
MODEL_REVISION = 'main' # @param {type:'string'}
LAYER = 14 # @param {type:'integer'}
N_PROBES = 8 # @param {type:'integer'}
MAX_NEW_TOKENS = 48 # @param {type:'integer'}

In [ ]:
OUTPUT = Path('/content/j7scope-output/traces')
CACHE = Path('/content/j7scope-cache/jacobian')
command = [
    sys.executable, 'experiments/capture_trace.py',
    '--trace-id', TRACE_ID,
    '--language', LANGUAGE,
    '--concept', CONCEPT,
    '--prompt', PROMPT,
    '--model', MODEL,
    '--model-revision', MODEL_REVISION,
    '--layer', str(LAYER),
    '--n-probes', str(N_PROBES),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--device', 'cuda',
    '--dtype', 'auto',
    '--out', str(OUTPUT),
    '--cache-dir', str(CACHE),
]
print('Starting capture. The first run downloads model weights and fits J_l...')
subprocess.run(command, check=True)

In [ ]:
subprocess.run([
    sys.executable, 'experiments/validate_trace_gallery.py', str(OUTPUT)
], check=True)

import shutil
archive = shutil.make_archive(
    f'/content/{TRACE_ID}', 'zip', root_dir=OUTPUT.parent, base_dir=OUTPUT.name
)
print(f'Validated archive: {archive}')

In [ ]:
from google.colab import files
files.download(archive)

## Submit

Follow [`CONTRIBUTING.md`](https://github.com/arthurpanhku/j7scope/blob/main/CONTRIBUTING.md): copy the trace directory into `results/traces/`, rebuild `index.json`, run validation, and open a pull request. Keep `preview: true` unless a maintainer has completed research review.